# 1. 대회 데이터 구조 확인

원본 `data/open.zip`을 수정하거나 압축 해제하지 않고 파일 구성과 dev 데이터 구조를 확인한다.

In [2]:
from pathlib import Path
import gzip, io, json, zipfile
import pandas as pd

ZIP_PATH = Path('../data/open.zip')
assert ZIP_PATH.exists(), ZIP_PATH
print(ZIP_PATH, ZIP_PATH.stat().st_size)

..\data\open.zip 216827912


### 2. 데이터 크기 및 데이터 파일명 확인

In [3]:
with zipfile.ZipFile(ZIP_PATH) as z:
    infos = z.infolist()
    for info in infos:
        print(f'{info.filename}\t{info.file_size:,} bytes')
print('파일 수:', len(infos))

README.md	7,176 bytes
baseline/requirements.txt	273 bytes
baseline/script.py	24,507 bytes
data/test.jsonl.gz	133,965 bytes
data/법령패키지/법령/(계약예규) 공동계약운용요령.txt	44,425 bytes
data/법령패키지/법령/(계약예규) 정부 입찰·계약 집행기준.txt	357,167 bytes
data/법령패키지/법령/국가를 당사자로 하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액.txt	4,095 bytes
data/법령패키지/법령/국가를 당사자로 하는 계약에 관한 법률 시행규칙.txt	253,770 bytes
data/법령패키지/법령/국가를 당사자로 하는 계약에 관한 법률 시행령.txt	331,790 bytes
data/법령패키지/법령/국가를 당사자로 하는 계약에 관한 법률.txt	51,478 bytes
data/법령패키지/법령/소상공인기본법 시행령.txt	31,101 bytes
data/법령패키지/법령/소상공인기본법.txt	29,231 bytes
data/법령패키지/법령/소프트웨어 진흥법 시행령.txt	100,223 bytes
data/법령패키지/법령/소프트웨어 진흥법.txt	88,410 bytes
data/법령패키지/법령/중소 소프트웨어사업자의 사업 참여 지원에 관한 지침.txt	244,210 bytes
data/법령패키지/법령/중소기업기본법 시행령.txt	91,044 bytes
data/법령패키지/법령/중소기업기본법.txt	45,740 bytes
data/법령패키지/법령/중소기업자간 경쟁제품 및 공사용자재 직접구매 대상 품목 지정 내역.txt	3,995 bytes
data/법령패키지/법령/중소기업자간 경쟁제품 직접생산 확인기준.txt	175,177 bytes
data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법률 시행규칙.txt	125,248 bytes
data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법

  ### 3. `dev.jsonl.gz` 데이터 읽기 및 구조 확인

  ### 확인 항목:
  - 전체 데이터 건수
  - 각 데이터의 주요 키
  - 첫 번째 데이터의 `id`
  - 데이터별 포함 문서 수
  - `meta` 메타데이터 구조

  - `id`: 데이터 식별자
  - `docs`: 참고 문서 목록
  - `meta`: 데이터 메타정보
  - `anon_applied`: 익명화 적용 여부
  - `assembly_policy_version`: 정책 버전
  - `dropped_doc_counts`: 제외된 문서 수
  - `input_completeness`: 입력 데이터 완전성


In [4]:
def load_jsonl_from_zip(zip_path, member, limit=None):
    rows = []
    with zipfile.ZipFile(zip_path) as z, z.open(member) as raw:
        with gzip.GzipFile(fileobj=raw) as stream:
            for line in stream:
                if line.strip():
                    rows.append(json.loads(line.decode('utf-8')))
                    if limit and len(rows) >= limit:
                        break
    return rows

dev = load_jsonl_from_zip(ZIP_PATH, 'dev.jsonl.gz')
print('dev rows:', len(dev))
print('top keys:', sorted(dev[0]))
print('first id:', dev[0]['id'])
print('docs:', len(dev[0]['docs']), 'meta keys:', len(dev[0]['meta']))

dev rows: 200
top keys: ['anon_applied', 'assembly_policy_version', 'docs', 'dropped_doc_counts', 'id', 'input_completeness', 'meta']
first id: PPS-DEV-01
docs: 2 meta keys: 21


### 4. dev 데이터 식별자, 문서 수, 문서 유형 확인
- id가 비어 있는 데이터가 있는지
- 중복된 id가 있는지
- 데이터마다 문서가 몇 개씩 포함되어 있는지
- 문서 유형에는 어떤 값이 있는지
- 데이터 누락 또는 중복 파악

In [5]:
ids = [r['id'] for r in dev]
print('id 결측:', sum(not x for x in ids))
print('id 중복:', len(ids) - len(set(ids)))
print('docs 개수 분포:', pd.Series([len(r['docs']) for r in dev]).value_counts().sort_index())
print('문서 유형:', sorted({d['type'] for r in dev for d in r['docs']}))

id 결측: 0
id 중복: 0
docs 개수 분포: 1    67
2    92
3    41
Name: count, dtype: int64
문서 유형: ['공고문', '과업지시서', '규격서', '제안요청서']


### 5. dev_labels.csv의 정답 라벨 확인

- 라벨 데이터의 행과 열 개수
- 라벨 컬럼 이름
- v1~v24 라벨 값의 종류
- id 중복 여부
- dev 라벨 파일을 연결가능한 지 , 라벨 형식 정상 여부 확인 

  - 총 200건의 라벨 데이터로 구성되어 있다.
  - 전체 컬럼은 49개이다.
    - `id`: 데이터 식별자
    - `v1~v24`: 24개의 이진 라벨
    - `e1~e24`: 24개의 추가 라벨
  - `v` 계열 라벨은 `0`과 `1`로 구성되어 있
  다.
  - `id` 중복은 발견되지 않았다.

In [7]:
with zipfile.ZipFile(ZIP_PATH) as z:
    labels = pd.read_csv(io.BytesIO(z.read('dev_labels.csv')))
print(labels.shape)
print(labels.columns.tolist())
print('v 값:', sorted(set(labels.filter(regex=r'^v\d+$').to_numpy().ravel())))
print('label id 중복:', labels['id'].duplicated().sum())

(200, 49)
['id', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'e9', 'e10', 'e11', 'e12', 'e13', 'e14', 'e15', 'e16', 'e17', 'e18', 'e19', 'e20', 'e21', 'e22', 'e23', 'e24']
v 값: [np.int64(0), np.int64(1)]
label id 중복: 0


## 6. 확인 결과 기록

- 원본 ZIP은 읽기 전용으로 사용한다.
- `dev.jsonl.gz`와 `dev_labels.csv`는 `id`로 대응한다.
- 다음 단계는 `--mock` 실행과 제출 CSV 검증이다.

In [28]:
def show_case(target_id):
    record = next(r for r in dev if r["id"] == target_id)
    answer = labels[labels["id"] == target_id].iloc[0]

    print("공고 ID:", target_id)
    print("문서:", [(d["type"], len(d["text"])) for d in
    record["docs"]])

    print("위반 항목:")
    for i in range(1, 25):
        if answer[f"v{i}"] == 1:
            print(f"v{i}:", answer[f"e{i}"])

In [35]:
for target_id in [r["id"] for r in dev[8:10]]:
      show_case(target_id)
      print("\n" + "-" * 60)

공고 ID: PPS-DEV-09
문서: [('공고문', 8770), ('규격서', 2493)]
위반 항목:
v7: 마. 주된 영업소(본사 소재지를 말함)가 입찰공고일 현재 광주광역시, 전라남도의 관할구역 안에 있어야 합니다.
v9: Chipset: GB10 Grace Blackwell Superchip

------------------------------------------------------------
공고 ID: PPS-DEV-10
문서: [('공고문', 10496), ('과업지시서', 7349)]
위반 항목:
v7: ③ 입찰공고일 전일부터 입찰일(낙찰자는 계약체결일)까지 법인등기부상 본점 소재지(개인사업자인 경우에는 사업자등록증 또는 관련 법령에 따른 허가·인가·면허·등록·신고 등에 관련된 서류에 기재된 사업장의 소재지)를 계속하여 전북특별자치도 또는 충청남도 또는 충청북도에 둔 업체여야 합니다.

------------------------------------------------------------


In [36]:
ids = [r["id"] for r in dev[:10]]

labels[labels["id"].isin(ids)].filter(regex=r"^v\d+$").sum()

v1     1
v2     2
v3     2
v4     1
v5     1
v6     1
v7     2
v8     1
v9     1
v10    0
v11    0
v12    0
v13    0
v14    0
v15    0
v16    0
v17    0
v18    0
v19    0
v20    0
v21    0
v22    0
v23    0
v24    0
dtype: int64

In [37]:
for i in range(1, 25):
    col = f"v{i}"
    found = labels[labels[col] == 1]

    if len(found) > 0:
        print(col, "대표 ID:", found.iloc[0]["id"])
    else:
        print(col, "위반 사례 없음")

v1 대표 ID: PPS-DEV-01
v2 대표 ID: PPS-DEV-02
v3 대표 ID: PPS-DEV-04
v4 대표 ID: PPS-DEV-06
v5 대표 ID: PPS-DEV-07
v6 대표 ID: PPS-DEV-08
v7 대표 ID: PPS-DEV-09
v8 대표 ID: PPS-DEV-05
v9 대표 ID: PPS-DEV-09
v10 대표 ID: PPS-DEV-13
v11 대표 ID: PPS-DEV-14
v12 대표 ID: PPS-DEV-15
v13 대표 ID: PPS-DEV-16
v14 대표 ID: PPS-DEV-17
v15 대표 ID: PPS-DEV-18
v16 대표 ID: PPS-DEV-20
v17 대표 ID: PPS-DEV-21
v18 대표 ID: PPS-DEV-22
v19 대표 ID: PPS-DEV-23
v20 대표 ID: PPS-DEV-24
v21 대표 ID: PPS-DEV-25
v22 대표 ID: PPS-DEV-26
v23 대표 ID: PPS-DEV-27
v24 대표 ID: PPS-DEV-29


### 7. 샘플 10개 파일 테스트

In [8]:
pilot = dev[:10]

pilot_ids = pd.DataFrame({
    "id": [r["id"] for r in pilot],
    "v1": "",
    "v2": "",
    "v3": "",
    "e1": "",
    "e2": "",
    "e3": "",
    "memo": "",
})

pilot_ids.to_csv(
    "C:/study-contests/outputs/labeling_pilot_10.csv",
    index=False,
    encoding="utf-8-sig",
)

pilot_ids

,id,v1,v2,v3,e1,e2,e3,memo
0,PPS-DEV-01,,,,,,,
1,PPS-DEV-02,,,,,,,
2,PPS-DEV-03,,,,,,,
3,PPS-DEV-04,,,,,,,
4,PPS-DEV-05,,,,,,,
5,PPS-DEV-06,,,,,,,
6,PPS-DEV-07,,,,,,,
7,PPS-DEV-08,,,,,,,
8,PPS-DEV-09,,,,,,,
9,PPS-DEV-10,,,,,,,


In [39]:
r = dev[0]
print("공고 ID:", r["id"])

for d in r["docs"]:
    print("\n문서 유형:", d["type"]) 
    print(d["text"])



공고 ID: PPS-DEV-01

문서 유형: 공고문
ㆍ[수요기관(공기업)] 공고 [공고번호]

[긴급] 용 역 입 찰 공 고
( 총액입찰, 제한경쟁(지역, 중소기업), 협상에 의한 계약 )

1. 입찰개요

구분 주요 내용 비고
‣ 용 역 명 : 2026 별바다부산 나이트 마켓 운영 대행 용역
입찰에 부치는 사항 ‣ 용역기간 : 계약체결일로부터 ~ 2026. 11. 30. 1항
‣ 기초금액 : 금310,000,000원(부가가치세 포함)

입찰 및 계약방법 ‣ 전자입찰 및 계약, 총액, 제한경쟁, 협상에 의한 계약, 상생결제 2항

‣ 업종제한 : 기타자유업(행사대행업,업종코드 : 9901)
‣ 직접생산 : 기타 행사기획 및 대행 서비스(세부품명번호: 8014199001)
입찰참가자격 축제기획및대행서비스 (세부품명번호: 9015189001) 4항
‣ 지역제한 : 부산광역시에 주된 영업소를 둔 업체
‣ 기업형태 : 중소기업, 소상공인

공동계약 가능여부 ‣ 공동계약 가능(공동이행) 4항

하도급 가능여부 ‣ 하도급 가능 4항

가격입찰서 제출 ‣ 2026. 2. 11. (수) 09:00 ~ 2026. 2. 19. (목) 12:00 ※ 전자제출 6항

제안서 제출 ‣ 2026. 2. 19 (목) 13:00 ~ 16:00 ※ 방문제출 7항

※ 입찰의 세부내용은 필히 본문 내용을 확인하시기 바랍니다.

2. 본 입찰에 참가하고자 하는 자는 국가종합전자조달시스템의 본 공고문과 함께 붙임
과업지시서, 제안요청서, 국가종합전자조달시스템 전자입찰특별유의서, 지방자치단체
입찰 및 계약집행기준, 계약일반조건 및 특수조건, 입찰유의서 등 기타 입찰에 필요
한 모든 사항을 열람 및 완전히 숙지하고 입찰에 응하셔야 하며, 이를 숙지하지 못
하여 발생한 불이익은 입찰참가자에게 있습니다.
<입찰 문의사항 안내>
☞ 서류 제출, 제안서 평가 관련 : [부서] 사업담당자 ☎[전화번호]
☞ 입찰 계약 체결 관련 : 인재개발팀 입찰담당자 ☎[전화번호]

3. 본 입찰은 청렴계약(서약)제가 적

###  각 v 항목의 대표 공고를 확인여 어떤 공고에서 1이 되는 지 파단

In [43]:
for i in range(1, 25):
    col = f"v{i}"
    rows = labels[labels[col] == 1]

    if len(rows) > 0:
        print(col, "대표 ID:", rows.iloc[0]["id"])

v1 대표 ID: PPS-DEV-01
v2 대표 ID: PPS-DEV-02
v3 대표 ID: PPS-DEV-04
v4 대표 ID: PPS-DEV-06
v5 대표 ID: PPS-DEV-07
v6 대표 ID: PPS-DEV-08
v7 대표 ID: PPS-DEV-09
v8 대표 ID: PPS-DEV-05
v9 대표 ID: PPS-DEV-09
v10 대표 ID: PPS-DEV-13
v11 대표 ID: PPS-DEV-14
v12 대표 ID: PPS-DEV-15
v13 대표 ID: PPS-DEV-16
v14 대표 ID: PPS-DEV-17
v15 대표 ID: PPS-DEV-18
v16 대표 ID: PPS-DEV-20
v17 대표 ID: PPS-DEV-21
v18 대표 ID: PPS-DEV-22
v19 대표 ID: PPS-DEV-23
v20 대표 ID: PPS-DEV-24
v21 대표 ID: PPS-DEV-25
v22 대표 ID: PPS-DEV-26
v23 대표 ID: PPS-DEV-27
v24 대표 ID: PPS-DEV-29


#### 대표 사례의 정답 근거 문장만 한 번에 확인

In [44]:
for i in range(1, 25):
    col = f"v{i}"
    rows = labels[labels[col] == 1]

    if len(rows) > 0:
        row = rows.iloc[0]
        evidence = row[f"e{i}"]

        print(f"\n[v{i}] {row['id']}")
        print("근거:", evidence)


[v1] PPS-DEV-01
근거: 라. 본 용역은 「고등교육법」 제2조에 따른 대학 또는 「산업교육진흥
및 산학연협력촉진에 관한 법률」 제25조에 따른 산학협력단만 참여 가능함

[v2] PPS-DEV-02
근거: 외국 학술지 공급계약을 3천만원 이상 체결한 실적이 있는 자

[v3] PPS-DEV-04
근거: 공고일 기준 최근 5년 간 단일 건으로 455,000,000원 이상(기초금액의 130% 이상)의 해양탐사 또는 수중조사 또는 침몰선박 조사·확인작업 분야 각종 작업 1건 이상 수행실적을 실적증명서로 제출이 가능한 업체

[v4] PPS-DEV-06
근거: 다. 최근 5년간 국가기관이 발주한 국제회의 개최 대행 용역의 이행실적이 4억원 이상인 업체

[v5] PPS-DEV-07
근거: 마. 본사(주된 영업소)의 소재지가 서울특별시에 있는 업체

[v6] PPS-DEV-08
근거: 라. 입찰공고일 전일부터 계약체결일까지 주된 영업소가 서울특별시 [지역:r1|단위=기초|광역=서울특별시] 내에 소재하고 있는 업체이어야 합니다.

[v7] PPS-DEV-09
근거: 마. 주된 영업소(본사 소재지를 말함)가 입찰공고일 현재 광주광역시, 전라남도의 관할구역 안에 있어야 합니다.

[v8] PPS-DEV-05
근거: 입찰공고일 전일 현재 제주특별자치도에 법인등기부상 본점 소재지(개인사업자인 경우에는 사업자등록증 또는 관련 법령에 따른 허가·인가·면허·등록·신고 등에 관련된 서류에 기재된 사업장의 소재지)를 두고 당해자격을 입찰일(낙찰자는 계약체결일)까지 유지하고 있는 업체로서 다음 각호의 자격을 모두 충족하여야 합니다.

[v9] PPS-DEV-09
근거: Chipset: GB10 Grace Blackwell Superchip

[v10] PPS-DEV-13
근거: nan

[v11] PPS-DEV-14
근거: nan

[v12] PPS-DEV-15
근거: 마. 본 입찰 대상 물품(과일류, 세부품명번호 5030990101)에 대한 직접생산확인증명서를 보유한 자이어

### 표로 정리하는 단계입니다.

In [ ]:
summary = []

for i in range(1, 25):
    col = f"v{i}"
    rows = labels[labels[col] == 1]

    if len(rows) > 0:
        row = rows.iloc[0]
        summary.append({
            "항목": col,
            "항목명": item_table["항목"][col]["항목명"],
            "대표 ID": row["id"],
            "근거": "" if pd.isna(row[f"e{i}"]) else row[f"e{i}"]
        })

summary_df = pd.DataFrame(summary)
pd.set_option("display.max_colwidth", None) # 긴 문장 자동으로 줄여서 보여줌
summary_df



,항목,항목명,대표 ID,근거
0,v1,참가자격 특정기관 제한,PPS-DEV-01,라. 본 용역은 「고등교육법」 제2조에 따른 대학 또는 「산업교육진흥\n및 산학연협력촉진에 관한 법률」 제25조에 따른 산학협력단만 참여 가능함
1,v2,고시금액 미만 실적제한,PPS-DEV-02,외국 학술지 공급계약을 3천만원 이상 체결한 실적이 있는 자
2,v3,실적제한 1배수 이상,PPS-DEV-04,"공고일 기준 최근 5년 간 단일 건으로 455,000,000원 이상(기초금액의 130% 이상)의 해양탐사 또는 수중조사 또는 침몰선박 조사·확인작업 분야 각종 작업 1건 이상 수행실적을 실적증명서로 제출이 가능한 업체"
3,v4,"고시금액 이상 특정기관, 특정실적",PPS-DEV-06,다. 최근 5년간 국가기관이 발주한 국제회의 개최 대행 용역의 이행실적이 4억원 이상인 업체
4,v5,고시금액 이상 지역제한,PPS-DEV-07,마. 본사(주된 영업소)의 소재지가 서울특별시에 있는 업체
5,v6,"고시금액 미만 지역제한 시,군,구",PPS-DEV-08,라. 입찰공고일 전일부터 계약체결일까지 주된 영업소가 서울특별시 [지역:r1|단위=기초|광역=서울특별시] 내에 소재하고 있는 업체이어야 합니다.
6,v7,고시금액 미만 지역제한 인접 확대,PPS-DEV-09,"마. 주된 영업소(본사 소재지를 말함)가 입찰공고일 현재 광주광역시, 전라남도의 관할구역 안에 있어야 합니다."
7,v8,중복제한 (실적+지역),PPS-DEV-05,입찰공고일 전일 현재 제주특별자치도에 법인등기부상 본점 소재지(개인사업자인 경우에는 사업자등록증 또는 관련 법령에 따른 허가·인가·면허·등록·신고 등에 관련된 서류에 기재된 사업장의 소재지)를 두고 당해자격을 입찰일(낙찰자는 계약체결일)까지 유지하고 있는 업체로서 다음 각호의 자격을 모두 충족하여야 합니다.
8,v9,과업지시서 특정 모델명 명시,PPS-DEV-09,Chipset: GB10 Grace Blackwell Superchip
9,v10,중기간 경쟁제품 입찰 직생 없음,PPS-DEV-13,


 ### v1~v3의 정확한 항목 정의를 확인
 

In [12]:
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in z.namelist():
        if name.endswith(".json"):
            print(name)

data/정답스키마_디코딩.json
data/항목표.json


### 항목표.json에서 v1~v24의 기준을 확인

In [15]:
for code, info in item_table["항목"].items():
    print(f"{code}: {info['항목명']}")
    print(f"  부재탐지: {info['부재탐지']}")
    print(f"  비고: {info['비고']}")
    print()



v1: 참가자격 특정기관 제한
  부재탐지: False
  비고: 

v2: 고시금액 미만 실적제한
  부재탐지: False
  비고: 지방 + 소액수의 가능

v3: 실적제한 1배수 이상
  부재탐지: False
  비고: 사업예산 기준

v4: 고시금액 이상 특정기관, 특정실적
  부재탐지: False
  비고: 특정기관 표현 다양

v5: 고시금액 이상 지역제한
  부재탐지: False
  비고: 지방, 지자체에 따라 고시금액 다름

v6: 고시금액 미만 지역제한 시,군,구
  부재탐지: False
  비고: 지방 + 소액수의 가능

v7: 고시금액 미만 지역제한 인접 확대
  부재탐지: False
  비고: 지방 + 소액수의 가능

v8: 중복제한 (실적+지역)
  부재탐지: False
  비고: 지방 + 소액수의 가능

v9: 과업지시서 특정 모델명 명시
  부재탐지: False
  비고: 

v10: 중기간 경쟁제품 입찰 직생 없음
  부재탐지: True
  비고: 

v11: 중기간 경쟁제품 입찰 중소 없음
  부재탐지: True
  비고: 

v12: 일반제품 직생 제한
  부재탐지: False
  비고: 중기간경쟁제품 고시 참고

v13: 중기간 경쟁제품 소기업, 소상공인 제한
  부재탐지: False
  비고: 

v14: 고시금액 이상 일반물품 중소기업 제한
  부재탐지: False
  비고: 

v15: 1억 이상- 고시금액미만 소기업 제한
  부재탐지: False
  비고: 판로지원 예외 명시한 경우, 제한 없어도 가능

v16: 1억 이상- 고시금액미만 중소기업 제한 없음
  부재탐지: True
  비고: 판로지원 예외 명시한 경우, 제한 없어도 가능

v17: 1억원 미만 일반물품 중소기업 제한
  부재탐지: False
  비고: 판로지원 예외 명시한 경우, 제한 없어도 가능

v18: 1억원 미만 일반물품 소기업 제한 없음
  부재탐지: True
  비고: 판로지원 예외 명시한 경우, 제한 없어도 가능

v19: 물품공급 확약서 입

In [16]:
#   법령 조문까지 모두 보고 싶으면:

for code, info in item_table["항목"].items():
    print(f"\n[{code}] {info['항목명']}")
    print("국가계약법:", info["국가계약법"])
    print("지방계약법:", info["지방계약법"])
    print("비고:", info["비고"])


[v1] 참가자격 특정기관 제한
국가계약법: 국가계약법 시행령 제12조 국가계약법 시행령 제21조
지방계약법: 지방계약법 시행령 제13조 지방계약법 시행령 제20조
비고: 

[v2] 고시금액 미만 실적제한
국가계약법: 국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조
지방계약법: 지방계약법 시행령 제20조 제1항 지방계약 법시행규칙제25조 제2항 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준 제1절 총칙 7. 계약담당자 주의사항
비고: 지방 + 소액수의 가능

[v3] 실적제한 1배수 이상
국가계약법: 국가계약법 시행령 제21조 제1항 국가계약법 시행규칙 제25조 (계약예규) 정부 입찰·계약 집행기준 제2장 제한경쟁입찰의 운용 제5조
지방계약법: 지방계약법 시행령 제20조 제1항 지방계약법 시행규칙 제25조
비고: 사업예산 기준

[v4] 고시금액 이상 특정기관, 특정실적
국가계약법: (계약예규) 정부입찰계약집행기준 제2장 제한경쟁입찰의 운용 제5조
지방계약법: (행안부예규) 지방자치단체 입찰 및 계약 집행기준 제1장 입찰 및 계약 일반기준
비고: 특정기관 표현 다양

[v5] 고시금액 이상 지역제한
국가계약법: 국가계약법 시행령 제21조 국가를 당사자로하는 계약에 관한 법률 등의 재정경제부장관이 정하는 고시금액
지방계약법: 지방계약법 시행령 제20조 지방계약법 시행규칙 제24조
비고: 지방, 지자체에 따라 고시금액 다름

[v6] 고시금액 미만 지역제한 시,군,구
국가계약법: 국가계약법 시행규칙 제25조
지방계약법: 지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영요령 제3절 수의계약 대상과 운영요령 1. 금액기준에 따른 2인 이상 견적서 제출 수의계약 나. 수의계약 요령
비고: 지방 + 소액수의 가능

[v7] 고시금액 미만 지역제한 인접 확대
국가계약법: 국가계약법 시행규칙 제25조
지방계약법: 지방계약법 시행규칙 제25조 지방자치단체 입찰 및 계약집행기준 제5장 수의계약 운영요령 제3